In [1]:
# Cell 1 — imports and config
import os
import time
import requests
import pandas as pd
import anthropic
from datetime import datetime, timedelta
from dotenv import load_dotenv

load_dotenv()

API_KEY       = os.getenv("NEWS_API_KEY")
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")
TOPIC         = '"Federal Reserve" AND ("interest rates" OR inflation)'
CSV_PATH      = "daily_sentiment.csv"

client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
print("Ready!")

/Users/ilynn/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Ready!


In [2]:
# Cell 2 — fetch headlines (day by day)
def fetch_headlines(topic, api_key, days_back=27):
    headlines = []
    url = "https://newsapi.org/v2/everything"
    now = datetime.now()

    for day_offset in range(days_back, 0, -1):
        day = now - timedelta(days=day_offset)
        day_str = day.strftime("%Y-%m-%d")

        params = {
            "q": topic,
            "from": day_str,
            "to": day_str,
            "language": "en",
            "sortBy": "publishedAt",
            "pageSize": 100,
            "apiKey": api_key,
        }

        response = requests.get(url, params=params)
        data = response.json()

        if data["status"] != "ok":
            print(f"Error on {day_str}:", data.get("message"))
            continue

        for article in data["articles"]:
            headlines.append({
                "title": article["title"],
                "published": article["publishedAt"][:10],
                "source": article["source"]["name"],
                "url": article["url"],
            })

        print(f"{day_str} — {len(data['articles'])} articles")
        time.sleep(0.5)

    return pd.DataFrame(headlines).drop_duplicates(subset="url")

print("fetch_headlines function ready!")

fetch_headlines function ready!


In [6]:
# Cell 3 — incremental fetch (fixed)
def fetch_incremental(topic, api_key, csv_path=CSV_PATH):

    if os.path.exists(csv_path):
        existing = pd.read_csv(csv_path)
        existing["published"] = pd.to_datetime(existing["published"]).dt.date.astype(str)
        last_date = existing["published"].max()
        print(f"Existing data found. Last date: {last_date}")
        print(f"Existing articles: {len(existing)}")

        # fetch from the day AFTER last date up to today
        last_dt = datetime.strptime(last_date, "%Y-%m-%d")
        days_since = (datetime.now() - last_dt).days - 1

        if days_since <= 0:
            print("Already up to date — skipping fetch")
            return existing

        print(f"Fetching {days_since} new days...")
    else:
        existing = pd.DataFrame()
        days_since = 27
        print("No existing data. Fetching full history...")

    new_df = fetch_headlines(topic, api_key, days_back=days_since)

    if new_df.empty:
        print("No new articles found")
        return existing

    combined = pd.concat([existing, new_df], ignore_index=True)
    combined = combined.drop_duplicates(subset="url")
    combined["published"] = pd.to_datetime(combined["published"]).dt.date.astype(str)
    combined = combined.sort_values("published").reset_index(drop=True)
    combined.to_csv(csv_path, index=False)

    print(f"New articles added: {len(combined) - len(existing)}")
    print(f"Total articles saved: {len(combined)}")
    return combined

df = fetch_incremental(TOPIC, API_KEY)

Existing data found. Last date: 2026-03-28
Existing articles: 1424
Already up to date — skipping fetch


In [7]:
# Cell 4 — score with FinBERT
from transformers import pipeline as hf_pipeline

print("Loading FinBERT...")
finbert = hf_pipeline("text-classification",
                       model="ProsusAI/finbert",
                       device=-1)

def score_finbert(title):
    try:
        result = finbert(str(title[:512]))[0]
        label = result["label"]
        score = result["score"]
        if label == "positive":   return score
        elif label == "negative": return -score
        else:                     return 0.0
    except:
        return 0.0

print("Scoring headlines...")
df["sentiment"] = df["title"].apply(score_finbert)
df["label"] = df["sentiment"].apply(
    lambda s: "positive" if s > 0.05 else "negative" if s < -0.05 else "neutral"
)
print(f"Done! Scored {len(df)} articles")

/Users/ilynn/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading FinBERT...


Device set to use cpu


Scoring headlines...
Done! Scored 1424 articles


In [8]:
# Cell 5 — aggregate and detect anomalies
daily = (
    df.groupby("published")
    .agg(
        avg_sentiment=("sentiment", "mean"),
        article_count=("title", "count"),
        positive=("label", lambda x: (x == "positive").sum()),
        negative=("label", lambda x: (x == "negative").sum()),
        neutral=("label", lambda x: (x == "neutral").sum()),
    )
    .reset_index()
)

daily["published"] = pd.to_datetime(daily["published"])
daily = daily.sort_values("published")

mean_vol     = daily["article_count"].mean()
std_vol      = daily["article_count"].std()
daily["heat_score"]  = (daily["article_count"] - mean_vol) / std_vol
daily["high_volume"] = daily["heat_score"] > 1.5

mean_sentiment = daily["avg_sentiment"].mean()
std_sentiment  = daily["avg_sentiment"].std()

anomalies = daily[
    (daily["high_volume"] == True) |
    (daily["avg_sentiment"] < mean_sentiment - 1.5 * std_sentiment)
].copy().sort_values("avg_sentiment")

print(f"Baseline sentiment: {mean_sentiment:.3f}")
print(f"Anomaly threshold:  {mean_sentiment - 1.5 * std_sentiment:.3f}")
print(f"Days flagged: {len(anomalies)}")
print()
for _, row in anomalies.iterrows():
    print(f"{row['published'].strftime('%b %d')} | "
          f"sentiment: {row['avg_sentiment']:.3f} | "
          f"articles: {row['article_count']} | "
          f"high volume: {row['high_volume']}")

Baseline sentiment: -0.335
Anomaly threshold:  -0.522
Days flagged: 3

Mar 22 | sentiment: -0.723 | articles: 11 | high volume: False
Mar 19 | sentiment: -0.379 | articles: 95 | high volume: True
Mar 18 | sentiment: -0.216 | articles: 96 | high volume: True


In [13]:
# Cell 6 — Claude investigation agent
def investigate_with_search(row):
    date_str = row["published"].strftime("%B %d, %Y")
    sentiment = row["avg_sentiment"]
    articles  = row["article_count"]

    messages = [{
        "role": "user",
        "content": f"""You are a senior financial analyst. Search the web for Federal Reserve and inflation news on {date_str}.

Context:
- Date: {date_str}
- Sentiment score: {sentiment:.3f} (-1.0 = very negative, +1.0 = very positive)
- Articles published: {articles}

After searching, respond in exactly this format:

KEYWORDS:
At most three keywords capturing the main themes

SUMMARY:
1-2 sentences explaining what happened and why sentiment hit {sentiment:.3f}

INVESTMENT WATCHOUT:
1-2 sentences on what investors should consider. Be specific about asset classes and sectors.

SOURCES:
List top three most prestige sources found as the name of channel"""
    }]

    max_turns = 6
    turns = 0
    final_report = ""

    while turns < max_turns:
        try:
            response = client.messages.create(
                model="claude-sonnet-4-20250514",
                max_tokens=700,
                tools=[{"type": "web_search_20250305", "name": "web_search"}],
                messages=messages
            )

            turns += 1

            for block in response.content:
                if hasattr(block, "text") and block.text:
                    final_report = block.text.strip()

            # strip whitespace from text blocks before appending
            cleaned_content = []
            for block in response.content:
                if hasattr(block, "text") and block.text:
                    block.text = block.text.strip()
                cleaned_content.append(block)
            messages.append({"role": "assistant", "content": cleaned_content})

            if response.stop_reason == "end_turn" and len(final_report) > 200:
                return final_report

            if response.stop_reason == "end_turn" and len(final_report) <= 200:
                messages.append({
                    "role": "user",
                    "content": "Please complete your full investigation report."
                })

            time.sleep(5)

        except anthropic.RateLimitError:
            print("Rate limit hit — waiting 60 seconds...")
            time.sleep(60)
            continue

    return final_report if final_report else "Investigation incomplete"

print("Investigation agent ready!")

Investigation agent ready!


In [14]:
# Cell 7 — run investigations and save reports
import json
from datetime import datetime as dt

reports = []

print("Running agentic investigation...\n")
print("=" * 60)

for _, row in anomalies.iterrows():
    date_str = row["published"].strftime("%B %d, %Y")
    print(f"\nInvestigating {date_str}...")
    print(f"Sentiment: {row['avg_sentiment']:.3f} | Articles: {row['article_count']}")
    print("-" * 40)

    report = investigate_with_search(row)
    print(report)
    print("=" * 60)

    reports.append({
        "date": date_str,
        "sentiment": round(row["avg_sentiment"], 3),
        "articles": int(row["article_count"]),
        "high_volume": bool(row["high_volume"]),
        "report": report,
        "investigated_at": dt.now().strftime("%Y-%m-%d %H:%M")
    })

    print("Waiting 60 seconds...")
    time.sleep(60)

# save reports to JSON
with open("investigation_reports.json", "w") as f:
    json.dump(reports, f, indent=2)

print(f"\nDone! Saved {len(reports)} reports to investigation_reports.json")

Running agentic investigation...


Investigating March 22, 2026...
Sentiment: -0.723 | Articles: 11
----------------------------------------
Rate limit hit — waiting 60 seconds...
This investigation confirms that the negative sentiment score of -0.723 on March 22, 2026 reflects deep market concerns about the intersection of geopolitical risk, energy-driven inflation, and Fed policy constraints in a challenging economic environment.
Waiting 60 seconds...

Investigating March 19, 2026...
Sentiment: -0.379 | Articles: 95
----------------------------------------
Rate limit hit — waiting 60 seconds...
Rate limit hit — waiting 60 seconds...
.

## SOURCES:
Federal Reserve
CNBC  
The Motley Fool

*Analysis based on 95 articles published March 19, 2026, with comprehensive cross-referencing of official Fed statements, market data, and geopolitical developments.*
Waiting 60 seconds...

Investigating March 18, 2026...
Sentiment: -0.216 | Articles: 96
----------------------------------------
Rate l